# Extract known variants for a gene from ClinVar database

## Requirements
### Python libraries
- pandas
- cyvcf2 (https://github.com/brentp/cyvcf2)

### CLI tools
- bedtools (https://bedtools.readthedocs.io/en/stable/)
- samtools (https://www.htslib.org/)

These tools are available as modules on OSC- if using bash magic in cells below, module must be loaded in each cell

```bash
module load bedtools2/2.31.0
module load samtools/1.21
```


In [1]:
# imports
import pandas as pd
import os
from cyvcf2 import VCF, Writer

pd.set_option('display.max_columns', None)

## common variables

In [2]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"


BASE_PATH = '/users/PAS2905/coraalbers/'
AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'

CLINVAR_PATH = '/users/PAS2905/coraalbers/ag/clinvar.vcf.gz'


# Download ClinVar dataset

In [ ]:
!curl -O https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_20260728.vcf.gz

# Get CDS intervals (GTF is 1-based inclusive → BED is 0-based half-open)

In [ ]:
%%bash
module load bedtools2/2.31.0
awk 'BEGIN{OFS="\t"} $3=="CDS" {print $1,$4-1,$5}' /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf | bedtools sort -i - | bedtools merge -i - > /users/PAS2905/coraalbers/ag/ag_data/cds.bed

**if needed, get chromosome sizes from fasta index file (only needs to be done once)**

In [ ]:
%%bash
# to create fasta index file (if needed)
module load samtools/1.21 
samtools faidx input.fasta

# to create chromosome sizes file (if needed)
cut -f1,2 /users/PAS2905/coraalbers/ag/hg38.fa.fai > hg38.chrom.sizes

# Extract noncoding regions (defined as complement of CDS)
**command needs to be run in directory with fasta index file and chromosome sizes file**


In [ ]:
%%bash
module load bedtools2/2.31.0
bedtools complement -i /users/PAS2905/coraalbers/ag/ag_data/cds.bed -g /users/PAS2905/coraalbers/ag/hg38.chrom.sizes > /users/PAS2905/coraalbers/ag/ag_data/noncoding.bed

# Remove chr from chromosome names to match the ClinVar VCF file chromosome naming scheme

In [ ]:
!sed 's/^chr//' noncoding.bed > noncoding_num_chr.bed

# Filter ClinVar database
filter to only LMNA affecting variants within 500 kb on either side of gene body (centered around gene)

In [3]:
PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic"    
}

BENIGN = {
    'Benign',
    'Likely_benign',
    'Benign/Likely_benign'
}

VUS = {
    'Uncertain_significance'
}


# define variant range (500 kb up and downstream from gene)
window_bp = 500000
vcf_range_start = LMNA_START - window_bp
vcf_range_end = LMNA_END + window_bp
vcf_range = f'1:{vcf_range_start}-{vcf_range_end}'


vcf = VCF(CLINVAR_PATH)

#### change variables here!!!
clinical_significance = VUS
output_vcf = "outputs/clinvar_LMNA.VUS.vcf"


In [4]:
# create a new vcf Writer using the input vcf as a template.
w = Writer(output_vcf, vcf)


for v in vcf(vcf_range):
    # print(v)
    clnsig = v.INFO.get("CLNSIG")
    # print(clnsig)
    if 'LMNA:' in v.INFO["GENEINFO"] and clnsig is not None:
        if any(item in v.INFO["CLNSIG"] for item in clinical_significance):
            w.write_record(v)


Benign
Benign
Benign
Benign
Likely_pathogenic
Likely_benign
Conflicting_classifications_of_pathogenicity
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Likely_benign
Likely_pathogenic
Pathogenic
Uncertain_significance
Uncertain_significance
Pathogenic/Likely_pathogenic
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Benign
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Conflicting_classifications_of_pathogenicity
Uncertain_significance
Conflicting_classifications_of_pathogenicity
Uncertain_significance
Conflicting_classifications_of_pathogenicity
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncertain_significance
Benign
Uncertain_significance
Uncertain_significance
Likely_pathogenic
Benign
Uncertain_significance
Uncertain_significance
Uncertain_significance
Uncer

# Use bedtools intersect to get variants that overlap with noncoding regions

In [ ]:
%%bash
module load bedtools2/2.31.0

bedtools intersect -wo -a /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/clinvar_LMNA.PLP.vcf -b /users/PAS2905/coraalbers/ag/ag_data/noncoding_num_chr.bed > outputs/plp_with_nc.bed
bedtools intersect -wo -a /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/clinvar_LMNA.BLB.vcf -b /users/PAS2905/coraalbers/ag/ag_data/noncoding_num_chr.bed > outputs/blb_with_nc.bed
bedtools intersect -wo -a /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/clinvar_LMNA.VUS.vcf -b /users/PAS2905/coraalbers/ag/ag_data/noncoding_num_chr.bed > outputs/vus_with_nc.bed